In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
pd.set_option("display.max_columns", None)

In [3]:
PROJECT_ROOT = Path("../../").resolve()
INPUT_PATH = PROJECT_ROOT/"ml"/"outputs"/"prepared_invoice_data.csv"
OUTPUT_PATH = PROJECT_ROOT/"ml"/"outputs"/"model_features.csv"

df = pd.read_csv(INPUT_PATH)

In [4]:
df.columns.tolist()

['invoice_id',
 'shipment_id',
 'load_id',
 'customer_id',
 'route_id',
 'contract_id',
 'invoice_date',
 'actual_distance_miles',
 'base_rate_per_mile',
 'calculated_base_charge',
 'fuel_surcharge_rate',
 'calculated_fuel_surcharge',
 'detention_minutes',
 'detention_free_minutes',
 'detention_rate_per_hour',
 'calculated_detention_charge',
 'calculated_accessorial_charges',
 'calculated_subtotal',
 'minimum_charge',
 'expected_total',
 'billed_amount',
 'load_id_shipment',
 'trip_id',
 'customer_id_shipment',
 'route_id_shipment',
 'load_date',
 'load_type',
 'weight_lbs',
 'pieces',
 'typical_distance_miles',
 'actual_distance_miles_shipment',
 'base_rate_per_mile_shipment',
 'fuel_surcharge_rate_shipment',
 'detention_minutes_shipment',
 'fuel_gallons_used',
 'actual_duration_hours',
 'average_mpg',
 'idle_time_hours',
 'customer_id_load',
 'route_id_load',
 'load_date_load',
 'load_type_load',
 'weight_lbs_load',
 'pieces_load',
 'revenue',
 'fuel_surcharge',
 'accessorial_charges

In [5]:
def safe_divide(numerator, denominator):
    """
    Safely divide two pandas Series.
    Returns NaN where denominator is zero or missing.
    """
    denominator = denominator.replace(0, np.nan)
    return numerator / denominator

In [6]:
df["distance_deviation"] = safe_divide(
    df["actual_distance_miles"] - df["typical_distance_miles"],
    df["typical_distance_miles"]
)
df["absolute_distance_deviation"] = (
    df["distance_deviation"].abs()
)

In [7]:
df["billed_per_mile"] = safe_divide(
    df["billed_amount"],
    df["actual_distance_miles"]
)

In [8]:
df["fuel_charge_per_mile"] = safe_divide(
    df["calculated_fuel_surcharge"],
    df["actual_distance_miles"]
)

In [9]:
df["accessorial_ratio"] = safe_divide(
    df["calculated_accessorial_charges"],
    df["billed_amount"]
)

In [10]:
duration_minutes = df["actual_duration_hours"] * 60

df["detention_ratio"] = safe_divide(
    df["detention_minutes"],
    duration_minutes
)

In [11]:
df["billed_to_expected_ratio"] = safe_divide(
    df["billed_amount"],
    df["expected_total"]
)

In [12]:
df["distance_to_duration_ratio"] = safe_divide(
    df["actual_distance_miles"],
    df["actual_duration_hours"]
)

In [13]:
df["fuel_efficiency"] = safe_divide(
    df["actual_distance_miles"],
    df["fuel_gallons_used"]
)

In [14]:
df["fuel_gallons_per_mile"] = safe_divide(
    df["fuel_gallons_used"],
    df["actual_distance_miles"]
)

In [15]:
df["idle_time_ratio"] = safe_divide(
    df["idle_time_hours"],
    df["actual_duration_hours"]
)

In [16]:
df["weight_per_piece"] = safe_divide(
    df["weight_lbs"],
    df["pieces"]
)

In [17]:
df["transportation_charge_per_mile"] = safe_divide(
    df["calculated_base_charge"],
    df["actual_distance_miles"]
)

In [18]:
df["fuel_surcharge_ratio"] = safe_divide(
    df["calculated_fuel_surcharge"],
    df["billed_amount"]
)

In [19]:
derived_features = [
    "distance_deviation",
    "absolute_distance_deviation",
    "billed_per_mile",
    "fuel_charge_per_mile",
    "accessorial_ratio",
    "detention_ratio",
    "billed_to_expected_ratio",
    "distance_to_duration_ratio",
    "fuel_efficiency",
    "fuel_gallons_per_mile",
    "idle_time_ratio",
    "weight_per_piece",
    "transportation_charge_per_mile",
    "fuel_surcharge_ratio"
]

df[derived_features].describe().T

,count,mean,std,min,25%,50%,75%,max
distance_deviation,85410.0,0.033844,0.039361,-0.023952,0.004975,0.030752,0.056424,0.241763
absolute_distance_deviation,85410.0,0.038024,0.035339,0.000000,0.012589,0.030752,0.056424,0.241763
billed_per_mile,85410.0,2.591958,0.463789,1.709996,2.231949,2.623524,2.874172,7.626344
fuel_charge_per_mile,85410.0,0.250157,0.062425,0.149956,0.190000,0.250000,0.300000,0.408005
accessorial_ratio,85410.0,0.033899,0.059320,0.000000,0.000000,0.016693,0.041454,0.714031
detention_ratio,85410.0,0.137727,0.267883,0.000000,0.024932,0.070175,0.141481,2.821429
billed_to_expected_ratio,85410.0,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000
distance_to_duration_ratio,85410.0,57.767441,4.601053,48.947368,53.874216,57.730982,61.497520,76.392857
fuel_efficiency,85410.0,6.531169,0.605237,5.487805,6.010861,6.529003,7.030427,8.628293
fuel_gallons_per_mile,85410.0,0.154432,0.014309,0.115898,0.142239,0.153163,0.166366,0.182222


In [20]:
feature_columns = [
    # Billing
    "billed_amount",
    "calculated_base_charge",
    "calculated_fuel_surcharge",
    "calculated_accessorial_charges",
    "calculated_detention_charge",
    "calculated_subtotal",
    "expected_total",

    # Distance
    "actual_distance_miles",
    "typical_distance_miles",
    "distance_deviation",
    "absolute_distance_deviation",

    # Billing intensity
    "base_rate_per_mile",
    "fuel_surcharge_rate",
    "billed_per_mile",
    "fuel_charge_per_mile",
    "transportation_charge_per_mile",
    "fuel_surcharge_ratio",
    "accessorial_ratio",

    # Operational behavior
    "detention_minutes",
    "detention_ratio",
    "actual_duration_hours",
    "distance_to_duration_ratio",
    "idle_time_hours",
    "idle_time_ratio",

    # Fuel
    "fuel_gallons_used",
    "average_mpg",
    "fuel_efficiency",
    "fuel_gallons_per_mile",

    # Shipment
    "weight_lbs",
    "pieces",
    "weight_per_piece",

    # Delivery evidence
    "delivery_event_count",
    "delivery_detention_minutes",
    "on_time_rate",

    # Billing relationship
    "billed_to_expected_ratio"
]

In [21]:
available_features = [
    col for col in feature_columns
    if col in df.columns
]

missing_features = [
    col for col in feature_columns
    if col not in df.columns
]

print("Available features:", len(available_features))
print("Missing features:", missing_features)

Available features: 35
Missing features: []


In [22]:
model_features = df[
    ["invoice_id"] + available_features
].copy()

print("Model feature table:", model_features.shape)

Model feature table: (85410, 36)


In [23]:
model_features[available_features].dtypes

billed_amount                     float64
calculated_base_charge            float64
calculated_fuel_surcharge         float64
calculated_accessorial_charges    float64
calculated_detention_charge       float64
calculated_subtotal               float64
expected_total                    float64
actual_distance_miles             float64
typical_distance_miles              int64
distance_deviation                float64
absolute_distance_deviation       float64
base_rate_per_mile                float64
fuel_surcharge_rate               float64
billed_per_mile                   float64
fuel_charge_per_mile              float64
transportation_charge_per_mile    float64
fuel_surcharge_ratio              float64
accessorial_ratio                 float64
detention_minutes                   int64
detention_ratio                   float64
actual_duration_hours             float64
distance_to_duration_ratio        float64
idle_time_hours                   float64
idle_time_ratio                   

In [24]:
for col in available_features:
    model_features[col] = pd.to_numeric(
        model_features[col],
        errors="coerce"
    )

In [ ]:
missing = (
    model_features[available_features]
    .isna()
    .sum()
    .sort_values(ascending=False)
)
missing[missing > 0]

Series([], dtype: int64)

In [26]:
numeric_values = model_features[available_features]

inf_count = np.isinf(numeric_values).sum()

inf_count[inf_count > 0]

Series([], dtype: int64)

In [27]:
model_features[available_features] = (
    model_features[available_features]
    .replace([np.inf, -np.inf], np.nan)
)

In [28]:
model_features[available_features].describe().T

,count,mean,std,min,25%,50%,75%,max
billed_amount,85410.0,3652.955994,2132.345520,250.000000,1761.202500,3412.500000,5003.330000,9948.180000
calculated_base_charge,85410.0,3187.713194,1930.800817,144.900000,1463.240000,2990.880000,4395.840000,8766.980000
calculated_fuel_surcharge,85410.0,365.119715,227.880760,13.500000,167.910000,340.560000,514.800000,1154.230000
calculated_accessorial_charges,85410.0,74.553624,73.512670,0.000000,0.000000,75.000000,150.000000,400.000000
calculated_detention_charge,85410.0,25.166207,35.652522,0.000000,0.000000,0.000000,49.170000,199.170000
calculated_subtotal,85410.0,3652.552740,2132.995469,160.200000,1761.202500,3412.500000,5003.330000,9948.180000
expected_total,85410.0,3652.955994,2132.345520,250.000000,1761.202500,3412.500000,5003.330000,9948.180000
actual_distance_miles,85410.0,1436.770610,806.531806,90.000000,718.000000,1301.000000,2153.000000,3898.500000
typical_distance_miles,85410.0,1389.270928,777.368131,92.000000,697.000000,1271.000000,2108.000000,3141.000000
distance_deviation,85410.0,0.033844,0.039361,-0.023952,0.004975,0.030752,0.056424,0.241763


In [ ]:
OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)
model_features.to_csv(
    OUTPUT_PATH,
    index=False
)

Saved: C:\Users\kshit\OneDrive\Desktop\Hack\ml\outputs\model_features.csv


In [30]:
print("Final feature table")
print("-" * 50)

print("Rows:", len(model_features))
print("Unique invoices:", model_features["invoice_id"].nunique())
print("ML features:", len(available_features))

assert model_features["invoice_id"].is_unique

print("\nOne row per invoice")
print("Feature engineering completed")
print("No model training performed")

Final feature table
--------------------------------------------------
Rows: 85410
Unique invoices: 85410
ML features: 35

One row per invoice
Feature engineering completed
No model training performed
